# BASELINE PREDICTIONS


In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score

In [3]:
df = pd.read_parquet("../data/processed/sales_feature_engineered.parquet")
df.head()

,city_id,store_id,product_id,dt,sale_amount,stock_hour6_22_cnt,activity_flag,discount,holiday_flag,precpt,...,week_of_year,quarter,is_weekend,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,0,0,54,2025-04-30,0.9,5,0,1.0,0,1.4621,...,18,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,0,54,2025-05-01,2.0,10,0,1.0,1,1.2773,...,18,2,0,0.9,NaN,NaN,NaN,NaN,NaN,NaN
2,0,0,54,2025-05-02,0.9,7,0,1.0,1,1.1190,...,18,2,0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
3,0,0,54,2025-05-03,1.0,0,0,1.0,1,1.8706,...,18,2,1,0.9,NaN,NaN,NaN,NaN,NaN,NaN
4,0,0,54,2025-05-04,1.2,9,0,1.0,1,2.1437,...,18,2,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print("Shape:", df.shape)
print("Date range:", df["dt"].min(), "to", df["dt"].max())
df.columns

Shape: (7869549, 27)
Date range: 2023-06-05 00:00:00 to 2025-07-13 00:00:00


Index(['city_id', 'store_id', 'product_id', 'dt', 'sale_amount',
       'stock_hour6_22_cnt', 'activity_flag', 'discount', 'holiday_flag',
       'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level', 'year',
       'month', 'day_of_month', 'day_of_week', 'week_of_year', 'quarter',
       'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7',
       'rolling_mean_14', 'rolling_mean_28'],
      dtype='str')

In [5]:
print("Number of unique dates:", df["dt"].nunique())
print("\nLast 10 dates:")
print(df["dt"].drop_duplicates().sort_values().tail(10).tolist())

Number of unique dates: 770

Last 10 dates:
[Timestamp('2025-07-04 00:00:00'), Timestamp('2025-07-05 00:00:00'), Timestamp('2025-07-06 00:00:00'), Timestamp('2025-07-07 00:00:00'), Timestamp('2025-07-08 00:00:00'), Timestamp('2025-07-09 00:00:00'), Timestamp('2025-07-10 00:00:00'), Timestamp('2025-07-11 00:00:00'), Timestamp('2025-07-12 00:00:00'), Timestamp('2025-07-13 00:00:00')]


In [6]:
n_dates = df["dt"].nunique()

train_dates = int(n_dates * 0.8)

unique_dates = np.sort(df["dt"].unique())

cutoff_date = unique_dates[train_dates]

print("Train dates:", train_dates)
print("Cutoff date:", cutoff_date)
print("Test dates:", n_dates - train_dates)

Train dates: 616
Cutoff date: 2025-02-10T00:00:00.000000
Test dates: 154


In [7]:
train = df[df["dt"] < cutoff_date].copy()
test = df[df["dt"] >= cutoff_date].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain range:")
print(train["dt"].min(), "to", train["dt"].max())

print("\nTest range:")
print(test["dt"].min(), "to", test["dt"].max())

Train shape: (6226148, 27)
Test shape: (1643401, 27)

Train range:
2023-06-05 00:00:00 to 2025-02-09 00:00:00

Test range:
2025-02-10 00:00:00 to 2025-07-13 00:00:00


In [8]:
test_baseline_1 = test.dropna(subset=["lag_1"]).copy()

test_baseline_1["baseline_lag1"] = test_baseline_1["lag_1"]

print("Rows available for baseline1:", len(test_baseline_1))
print("Rows excluded due to missing lag_1:", test["lag_1"].isna().sum())

Rows available for baseline1: 1639185
Rows excluded due to missing lag_1: 4216


In [9]:
test_baseline_1[["store_id", "product_id", "dt", "sale_amount", "lag_1", "baseline_lag1"]].head(10)

,store_id,product_id,dt,sale_amount,lag_1,baseline_lag1
1,0,54,2025-05-01,2.0,0.9,0.9
2,0,54,2025-05-02,0.9,2.0,2.0
3,0,54,2025-05-03,1.0,0.9,0.9
4,0,54,2025-05-04,1.2,1.0,1.0
5,0,54,2025-05-05,1.3,1.2,1.2
6,0,54,2025-05-06,0.7,1.3,1.3
7,0,54,2025-05-07,1.2,0.7,0.7
8,0,54,2025-05-08,1.5,1.2,1.2
9,0,54,2025-05-09,1.4,1.5,1.5
10,0,54,2025-05-10,1.7,1.4,1.4


In [10]:
y_true = test_baseline_1["sale_amount"]
y_pred = test_baseline_1["baseline_lag1"]

mae_lag1 = mean_absolute_error(y_true, y_pred)
rmse_lag1 = np.sqrt(mean_squared_error(y_true, y_pred))
r2_lag1 = r2_score(y_true, y_pred)

print("Baseline 1 - Previous Day")
print("-------------------------")
print("MAE  :", mae_lag1)
print("RMSE :", rmse_lag1)
print("R² :", r2_lag1)


Baseline 1 - Previous Day
-------------------------
MAE  : 0.5998853295997706
RMSE : 0.94415100980257
R² : 0.5099986660711421


In [11]:
test_baseline_7 = test.dropna(subset=["lag_7"]).copy()

test_baseline_7["baseline_lag7"] = test_baseline_7["lag_7"]

print("Rows available:", len(test_baseline_7))
print("Rows excluded:", test["lag_7"].isna().sum())

Rows available: 1612608
Rows excluded: 30793


In [12]:
y_true_7 = test_baseline_7["sale_amount"]
y_pred_7 = test_baseline_7["baseline_lag7"]

mae_lag7 = mean_absolute_error(y_true_7, y_pred_7)
rmse_lag7 = np.sqrt(mean_squared_error(y_true_7, y_pred_7))
r2_lag7 = r2_score(y_true_7, y_pred_7)

print("Baseline 2 - Same Day Last Week")
print("--------------------------------")
print("MAE  :", mae_lag7)
print("RMSE :", rmse_lag7)
print("R²   :", r2_lag7)

Baseline 2 - Same Day Last Week
--------------------------------
MAE  : 0.6287254943544865
RMSE : 1.0089156135383166
R²   : 0.4442816614689302


# LIGHTGBM

In [13]:
# ============================================================
# LIGHTGBM - FEATURE PREPARATION
# ============================================================

import lightgbm as lgbm
import joblib

print("LightGBM version:", lgbm.__version__)

excluded = ["dt", "stock_hour6_22_cnt", "sale_amount"]

lgb_features = [
    col for col in df.columns
    if col not in excluded
]

print("Number of features:", len(lgb_features))
print(lgb_features)

x_train = train[lgb_features].copy()
y_train = train["sale_amount"]

x_test = test[lgb_features].copy()
y_test = test["sale_amount"]

print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)

cat_cols = ["city_id", "store_id", "product_id"]

for col in cat_cols:
    x_train[col] = x_train[col].astype("category")
    x_test[col] = x_test[col].astype("category")

LightGBM version: 4.7.0
Number of features: 24
['city_id', 'store_id', 'product_id', 'activity_flag', 'discount', 'holiday_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level', 'year', 'month', 'day_of_month', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28']
Train: (6226148, 24) (6226148,)
Test : (1643401, 24) (1643401,)


In [14]:
# ============================================================
# INITIAL LIGHTGBM BASELINE
# ============================================================

lgbm_model = lgbm.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(
    x_train,
    y_train,
    categorical_feature=cat_cols
)

y_pred_lgbm = lgbm_model.predict(x_test)

mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)
rmse_lgbm = np.sqrt(mean_squared_error(y_test, y_pred_lgbm))
r2_lgbm = r2_score(y_test, y_pred_lgbm)

print("\n" + "=" * 60)
print("INITIAL LIGHTGBM BASELINE")
print("=" * 60)

print("MAE  :", mae_lgbm)
print("RMSE :", rmse_lgbm)
print("R²   :", r2_lgbm)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.295000 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4388
[LightGBM] [Info] Number of data points in the train set: 6226148, number of used features: 24
[LightGBM] [Info] Start training from score 1.735404

INITIAL LIGHTGBM BASELINE
MAE  : 0.4868930431962395
RMSE : 0.7435994943422592
R²   : 0.6956769337538944


In [15]:
# ============================================================
# TIME-BASED VALIDATION SPLIT
# ============================================================

train_dates = np.sort(train["dt"].unique())

val_start = train_dates[int(len(train_dates) * 0.8)]

tune_train = train[train["dt"] < val_start]
val = train[train["dt"] >= val_start]

print("Tune train:", tune_train["dt"].min(), "to", tune_train["dt"].max())
print("Validation:", val["dt"].min(), "to", val["dt"].max())

print("Tune train shape:", tune_train.shape)
print("Validation shape:", val.shape)

X_tune = tune_train[lgb_features].copy()
y_tune = tune_train["sale_amount"]

X_val = val[lgb_features].copy()
y_val = val["sale_amount"]

for col in cat_cols:
    X_tune[col] = X_tune[col].astype("category")
    X_val[col] = X_val[col].astype("category")

Tune train: 2023-06-05 00:00:00 to 2024-10-08 00:00:00
Validation: 2024-10-09 00:00:00 to 2025-02-09 00:00:00
Tune train shape: (4953452, 27)
Validation shape: (1272696, 27)


In [16]:
# ============================================================
# LIGHTGBM HYPERPARAMETER TUNING
# ============================================================

param_grid = [
    {"num_leaves": 31, "learning_rate": 0.03, "n_estimators": 800, "max_depth": -1},
    {"num_leaves": 63, "learning_rate": 0.03, "n_estimators": 800, "max_depth": -1},
    {"num_leaves": 127, "learning_rate": 0.03, "n_estimators": 800, "max_depth": -1},
    {"num_leaves": 63, "learning_rate": 0.02, "n_estimators": 1000, "max_depth": -1},
    {"num_leaves": 127, "learning_rate": 0.02, "n_estimators": 1000, "max_depth": -1},
    {"num_leaves": 63, "learning_rate": 0.05, "n_estimators": 800, "max_depth": 15},
    {"num_leaves": 127, "learning_rate": 0.05, "n_estimators": 800, "max_depth": 20},
    {"num_leaves": 63, "learning_rate": 0.03, "n_estimators": 1000, "max_depth": 15},
    {"num_leaves": 127, "learning_rate": 0.03, "n_estimators": 1000, "max_depth": 20},
    {"num_leaves": 31, "learning_rate": 0.02, "n_estimators": 1000, "max_depth": -1}
]

results = []

for i, params in enumerate(param_grid, 1):

    print(f"Training model {i}/{len(param_grid)}: {params}")

    temp_model = lgbm.LGBMRegressor(
        objective="regression",
        random_state=42,
        n_jobs=-1,
        **params
    )

    temp_model.fit(
        X_tune,
        y_tune,
        categorical_feature=cat_cols
    )

    val_pred = temp_model.predict(X_val)

    mae = mean_absolute_error(y_val, val_pred)
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    r2 = r2_score(y_val, val_pred)

    results.append({
        **params,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("\n" + "=" * 60)
print("HYPERPARAMETER TUNING RESULTS")
print("=" * 60)

print(results_df)

Training model 1/10: {'num_leaves': 31, 'learning_rate': 0.03, 'n_estimators': 800, 'max_depth': -1}
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.196778 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4354
[LightGBM] [Info] Number of data points in the train set: 4953452, number of used features: 24
[LightGBM] [Info] Start training from score 1.846085
Training model 2/10: {'num_leaves': 63, 'learning_rate': 0.03, 'n_estimators': 800, 'max_depth': -1}
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin a

In [17]:
# ============================================================
# SELECT BEST HYPERPARAMETERS
# ============================================================

best_row = results_df.iloc[0]

best_params = {
    "num_leaves": int(best_row["num_leaves"]),
    "learning_rate": float(best_row["learning_rate"]),
    "n_estimators": int(best_row["n_estimators"]),
    "max_depth": int(best_row["max_depth"])
}

print("Best parameters:")
print(best_params)

Best parameters:
{'num_leaves': 127, 'learning_rate': 0.02, 'n_estimators': 1000, 'max_depth': -1}


In [18]:
# ============================================================
# FINAL OPTIMIZED LIGHTGBM
# ============================================================

best_model = lgbm.LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1,
    **best_params
)

# Train final model on COMPLETE training data
best_model.fit(
    x_train,
    y_train,
    categorical_feature=cat_cols
)

y_pred_best = best_model.predict(x_test)

mae_best = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best = r2_score(y_test, y_pred_best)

print("\n" + "=" * 60)
print("FINAL OPTIMIZED LIGHTGBM")
print("=" * 60)

print("MAE  :", mae_best)
print("RMSE :", rmse_best)
print("R²   :", r2_best)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.267352 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4388
[LightGBM] [Info] Number of data points in the train set: 6226148, number of used features: 24
[LightGBM] [Info] Start training from score 1.735404

FINAL OPTIMIZED LIGHTGBM
MAE  : 0.470521864354885
RMSE : 0.7263556491103703
R²   : 0.7096275983673721


In [22]:
test_out = test[['store_id', 'product_id', 'sale_amount']].copy()
test_out['prediction'] = y_pred_best
test_out.to_csv('test_predictions.csv', index=False)

print("Saved test_predictions.csv:", test_out.shape)
test_out.head()

Saved test_predictions.csv: (1643401, 4)


,store_id,product_id,sale_amount,prediction
0,0,54,0.9,0.558921
1,0,54,2.0,1.071435
2,0,54,0.9,1.884191
3,0,54,1.0,1.188964
4,0,54,1.2,1.231826


In [19]:
# ============================================================
# FINAL FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "feature": x_train.columns,
    "importance": best_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance)

importance.to_csv(
    "lightgbm_feature_importance.csv",
    index=False
)

print("Feature importance saved.")

            feature  importance
1          store_id       21305
2        product_id       20309
17            lag_1       10271
12     day_of_month        9498
7   avg_temperature        6658
18            lag_7        6144
13      day_of_week        5769
6            precpt        5554
14     week_of_year        5485
21   rolling_mean_7        4890
19           lag_14        4758
8      avg_humidity        4600
20           lag_28        4156
4          discount        3315
9    avg_wind_level        2999
5      holiday_flag        2193
23  rolling_mean_28        2153
0           city_id        1854
22  rolling_mean_14        1835
11            month        1286
10             year         863
3     activity_flag          54
16       is_weekend          44
15          quarter           7
Feature importance saved.


In [20]:
# ============================================================
# SAVE FINAL LIGHTGBM MODEL
# ============================================================

joblib.dump(
    best_model,
    "best_lightgbm.pkl"
)

joblib.dump(
    x_train.columns.tolist(),
    "lgbm_features.pkl"
)

print("LightGBM model saved: best_lightgbm.pkl")
print("Feature list saved: lgbm_features.pkl")

LightGBM model saved: best_lightgbm.pkl
Feature list saved: lgbm_features.pkl


In [21]:
# ============================================================
# FINAL VERIFICATION
# ============================================================

print("Model:", type(best_model))

print("\nTrain shape:", x_train.shape)
print("Test shape :", x_test.shape)

print("\nTest date range:")
print(test["dt"].min(), "to", test["dt"].max())

print("\nFeature count:", len(x_test.columns))

print("\nFeature columns:")
print(x_test.columns.tolist())

Model: <class 'lightgbm.sklearn.LGBMRegressor'>

Train shape: (6226148, 24)
Test shape : (1643401, 24)

Test date range:
2025-02-10 00:00:00 to 2025-07-13 00:00:00

Feature count: 24

Feature columns:
['city_id', 'store_id', 'product_id', 'activity_flag', 'discount', 'holiday_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level', 'year', 'month', 'day_of_month', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28']


In [23]:
import pandas as pd

test_predictions = pd.read_csv(r"C:\Users\sneha\SupplyChainForecasting\models\test_predictions.csv")

print(test_predictions.dtypes)
print(test_predictions[["store_id", "product_id"]].drop_duplicates().shape)
print(test_predictions[(test_predictions["store_id"] == 0) & (test_predictions["product_id"] == 474)])
print(test_predictions[test_predictions["store_id"].astype(str) == "0"]["product_id"].astype(str).eq("474").sum())

store_id         int64
product_id       int64
sale_amount    float64
prediction     float64
dtype: object
(14351, 2)
Empty DataFrame
Columns: [store_id, product_id, sale_amount, prediction]
Index: []
0


# LSTM Model

In [21]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.12.0+cpu
Device: cpu


In [22]:
df = pd.read_parquet(
    r"C:\Users\sneha\SupplyChainForecasting\data\processed\sales_feature_engineered.parquet"
)
print("Shape:", df.shape)
print("Date range:", df["dt"].min(), "to", df["dt"].max())


Shape: (7869549, 27)
Date range: 2023-06-05 00:00:00 to 2025-07-13 00:00:00


In [23]:
TARGET = "sale_amount"

CONTINUOUS_FEATURES = [
    "activity_flag", "discount", "holiday_flag", "precpt",
    "avg_temperature", "avg_humidity", "avg_wind_level",
    "year", "month", "day_of_month", "day_of_week",
    "week_of_year", "quarter", "is_weekend",
]

CATEGORICAL_FEATURES = ["city_id", "store_id", "product_id"]

WINDOW = 15

print("Continuous features:", len(CONTINUOUS_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
assert len(CONTINUOUS_FEATURES) == 14

Continuous features: 14
Categorical features: 3


In [24]:
df = df.sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True)
N = len(df)
print("Sorted rows:", N)


Sorted rows: 7869549


In [25]:
TRAIN_END = pd.Timestamp("2025-02-09")
TEST_START = pd.Timestamp("2025-02-10")

train_mask_dates = df["dt"] <= TRAIN_END
test_mask = (df["dt"] >= TEST_START).to_numpy()

train_dates = np.sort(df.loc[train_mask_dates, "dt"].unique())
val_start = train_dates[int(len(train_dates) * 0.80)]

train_mask = (df["dt"] < val_start).to_numpy()
val_mask = ((df["dt"] >= val_start) & (df["dt"] <= TRAIN_END)).to_numpy()
# test_mask already computed above

print("Train :", df.loc[train_mask, "dt"].min(), "to", df.loc[train_mask, "dt"].max(),
      "| rows:", train_mask.sum())
print("Val   :", df.loc[val_mask, "dt"].min(), "to", df.loc[val_mask, "dt"].max(),
      "| rows:", val_mask.sum())
print("Test  :", df.loc[test_mask, "dt"].min(), "to", df.loc[test_mask, "dt"].max(),
      "| rows:", test_mask.sum())


Train : 2023-06-05 00:00:00 to 2024-10-08 00:00:00 | rows: 4953452
Val   : 2024-10-09 00:00:00 to 2025-02-09 00:00:00 | rows: 1272696
Test  : 2025-02-10 00:00:00 to 2025-07-13 00:00:00 | rows: 1643401


In [26]:
scaler = StandardScaler()
scaler.fit(df.loc[train_mask, CONTINUOUS_FEATURES])

cont_arr = scaler.transform(df[CONTINUOUS_FEATURES]).astype(np.float32)
print("cont_arr:", cont_arr.shape)

joblib.dump(scaler, "lstm_scaler.pkl")
print("Scaler saved: lstm_scaler.pkl")

cont_arr: (7869549, 14)
Scaler saved: lstm_scaler.pkl


In [27]:
cat_arr = df[CATEGORICAL_FEATURES].to_numpy(dtype=np.int64)
y_arr = df[TARGET].to_numpy(dtype=np.float32)

NUM_CITIES = int(df["city_id"].max()) + 1
NUM_STORES = int(df["store_id"].max()) + 1
NUM_PRODUCTS = int(df["product_id"].max()) + 1
print("Embedding sizes -> city:", NUM_CITIES, "store:", NUM_STORES, "product:", NUM_PRODUCTS)

Embedding sizes -> city: 18 store: 1057 product: 576


In [28]:
day_gap = df.groupby(["store_id", "product_id"])["dt"].diff().dt.days

new_segment = (
    (df["store_id"] != df["store_id"].shift(1)) |
    (df["product_id"] != df["product_id"].shift(1)) |
    (day_gap > 1)
)
segment_id = new_segment.cumsum().to_numpy()

print("Number of continuous segments:", len(np.unique(segment_id)))


Number of continuous segments: 35440


In [29]:
valid_target = np.zeros(N, dtype=bool)
valid_target[WINDOW:] = segment_id[WINDOW:] == segment_id[:-WINDOW]

train_indices = np.where(valid_target & train_mask)[0]
val_indices = np.where(valid_target & val_mask)[0]
test_indices = np.where(valid_target & test_mask)[0]

print("Usable sequences -> train:", len(train_indices),
      "val:", len(val_indices), "test:", len(test_indices))


Usable sequences -> train: 4567721 val: 1215600 test: 1576934


In [30]:
 #Dataset / DataLoader
class SalesLSTMDataset(Dataset):
    def __init__(self, cont, cat, target, indices, window=WINDOW):
        self.cont = cont
        self.cat = cat
        self.target = target
        self.indices = indices
        self.window = window

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        target_idx = self.indices[idx]
        start_idx = target_idx - self.window
        end_idx = target_idx

        x_cont = self.cont[start_idx:end_idx]
        x_cat = self.cat[start_idx:end_idx]
        y = self.target[target_idx]

        return (
            torch.from_numpy(x_cont),
            torch.from_numpy(x_cat),
            torch.tensor(y, dtype=torch.float32),
        )


BATCH_SIZE = 256

train_dataset = SalesLSTMDataset(cont_arr, cat_arr, y_arr, train_indices)
val_dataset = SalesLSTMDataset(cont_arr, cat_arr, y_arr, val_indices)
test_dataset = SalesLSTMDataset(cont_arr, cat_arr, y_arr, test_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Batches -> train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))

Batches -> train: 17843 val: 4749 test: 6160


In [31]:
class LSTMForecast(nn.Module):
    def __init__(self, num_cities, num_stores, num_products,
                 city_emb_dim=4, store_emb_dim=16, product_emb_dim=16,
                 continuous_dim=14, hidden_dim=64):
        super().__init__()
        self.city_emb = nn.Embedding(num_cities, city_emb_dim)
        self.store_emb = nn.Embedding(num_stores, store_emb_dim)
        self.product_emb = nn.Embedding(num_products, product_emb_dim)

        input_dim = continuous_dim + city_emb_dim + store_emb_dim + product_emb_dim
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim,
                             num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x_cont, x_cat):
        city = self.city_emb(x_cat[:, :, 0])
        store = self.store_emb(x_cat[:, :, 1])
        product = self.product_emb(x_cat[:, :, 2])

        x = torch.cat([x_cont, city, store, product], dim=-1)
        out, _ = self.lstm(x)
        last_step = out[:, -1, :]
        return self.fc(last_step).squeeze(-1)


model = LSTMForecast(NUM_CITIES, NUM_STORES, NUM_PRODUCTS).to(device)
print(model)

LSTMForecast(
  (city_emb): Embedding(18, 4)
  (store_emb): Embedding(1057, 16)
  (product_emb): Embedding(576, 16)
  (lstm): LSTM(50, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [32]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [33]:
EPOCHS = 10
PATIENCE = 2

best_val_loss = float("inf")
best_state = None
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    # ---- Train ----
    model.train()
    train_loss, train_count = 0.0, 0

    for x_cont, x_cat, y in train_loader:
        x_cont, x_cat, y = x_cont.to(device), x_cat.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(x_cont, x_cat)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        bs = y.size(0)
        train_loss += loss.item() * bs
        train_count += bs

    train_loss /= train_count

    # ---- Validate ----
    model.eval()
    val_loss, val_count = 0.0, 0

    with torch.no_grad():
        for x_cont, x_cat, y in val_loader:
            x_cont, x_cat, y = x_cont.to(device), x_cat.to(device), y.to(device)
            preds = model(x_cont, x_cat)
            loss = criterion(preds, y)
            bs = y.size(0)
            val_loss += loss.item() * bs
            val_count += bs
            
    val_loss /= val_count
            
    print(f"Epoch {epoch}/{EPOCHS} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")
            
    if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print("   -> best model saved")
    else:
             patience_counter += 1
             print(f"   -> no improvement ({patience_counter}/{PATIENCE})")
             if patience_counter >= PATIENCE:
                     print(f"Early stopping at epoch {epoch}.")
                     break
            
print("Best validation MSE:", best_val_loss)

C:\Users\sneha\AppData\Local\Temp\ipykernel_25916\1651981334.py:24: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(x_cat),


Epoch 1/10 | Train MSE: 1.363190 | Val MSE: 1.073644
   -> best model saved
Epoch 2/10 | Train MSE: 0.912716 | Val MSE: 1.097940
   -> no improvement (1/2)
Epoch 3/10 | Train MSE: 0.828210 | Val MSE: 1.141682
   -> no improvement (2/2)
Early stopping at epoch 3.
Best validation MSE: 1.0736443469956072


In [34]:
model.load_state_dict(best_state)
model.eval()

torch.save(best_state, "best_lstm.pth")
print("Best LSTM weights saved: best_lstm.pth")

Best LSTM weights saved: best_lstm.pth


In [35]:
def collect_predictions(loader):
    preds_all, actuals_all = [], []
    with torch.no_grad():
        for x_cont, x_cat, y in loader:
            x_cont, x_cat = x_cont.to(device), x_cat.to(device)
            preds = model(x_cont, x_cat)
            preds_all.append(preds.cpu().numpy())
            actuals_all.append(y.numpy())
    return np.concatenate(preds_all), np.concatenate(actuals_all)


val_preds, val_actuals = collect_predictions(val_loader)
val_mae = mean_absolute_error(val_actuals, val_preds)
val_rmse = np.sqrt(mean_squared_error(val_actuals, val_preds))
val_r2 = r2_score(val_actuals, val_preds)

print(f"Validation MAE  : {val_mae:.6f}")
print(f"Validation RMSE : {val_rmse:.6f}")
print(f"Validation R2   : {val_r2:.6f}")

Validation MAE  : 0.746668
Validation RMSE : 1.036168
Validation R2   : 0.309024


In [36]:
test_preds, test_actuals = collect_predictions(test_loader)
test_mae = mean_absolute_error(test_actuals, test_preds)
test_rmse = np.sqrt(mean_squared_error(test_actuals, test_preds))
test_r2 = r2_score(test_actuals, test_preds)

print(f"Test MAE  : {test_mae:.6f}")
print(f"Test RMSE : {test_rmse:.6f}")
print(f"Test R2   : {test_r2:.6f}")

Test MAE  : 0.856101
Test RMSE : 1.268904
Test R2   : 0.130103


In [37]:
# ============================================================
# LIGHTGBM SCORED ON THE SAME TEST ROWS AS THE LSTM (fair comparison)
# ============================================================
lgbm_test_subset = df.iloc[test_indices]
x_test_matched = lgbm_test_subset[lgb_features].copy()
for col in cat_cols:
    x_test_matched[col] = x_test_matched[col].astype("category")
y_test_matched = lgbm_test_subset[TARGET]

y_pred_matched = best_model.predict(x_test_matched)

mae_matched = mean_absolute_error(y_test_matched, y_pred_matched)
rmse_matched = np.sqrt(mean_squared_error(y_test_matched, y_pred_matched))
r2_matched = r2_score(y_test_matched, y_pred_matched)

print("LightGBM (scored on LSTM's matched test_indices):")
print(f"MAE: {mae_matched:.6f}  RMSE: {rmse_matched:.6f}  R2: {r2_matched:.6f}")

# ============================================================
# CONSOLIDATED FINAL RESULTS TABLE
# ============================================================
final_results = pd.DataFrame([
    {"Model": "Baseline - Lag 1",        "Split": "Test", "MAE": mae_lag1,  "RMSE": rmse_lag1,  "R2": r2_lag1,  "N": len(test_baseline_1)},
    {"Model": "Baseline - Lag 7",        "Split": "Test", "MAE": mae_lag7,  "RMSE": rmse_lag7,  "R2": r2_lag7,  "N": len(test_baseline_7)},
    {"Model": "LightGBM (full test set)","Split": "Test", "MAE": mae_best,  "RMSE": rmse_best,  "R2": r2_best,  "N": len(x_test)},
    {"Model": "LightGBM (matched to LSTM rows)", "Split": "Test", "MAE": mae_matched, "RMSE": rmse_matched, "R2": r2_matched, "N": len(x_test_matched)},
    {"Model": "LSTM",                    "Split": "Validation", "MAE": val_mae, "RMSE": val_rmse, "R2": val_r2, "N": len(val_indices)},
    {"Model": "LSTM",                    "Split": "Test", "MAE": test_mae, "RMSE": test_rmse, "R2": test_r2, "N": len(test_indices)},
])

print("\n" + "=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)
print(final_results.to_string(index=False))

final_results.to_csv("final_model_comparison.csv", index=False)
print("\nSaved: final_model_comparison.csv")

# ============================================================
# RUN METADATA (for reproducibility / deployment reference)
# ============================================================
import json

run_metadata = {
    "data_shape": list(df.shape),
    "date_range": [str(df["dt"].min()), str(df["dt"].max())],
    "train_test_cutoff": str(cutoff_date),
    "lstm_train_end": str(TRAIN_END),
    "lstm_test_start": str(TEST_START),
    "lstm_window": WINDOW,
    "lgb_features": lgb_features,
    "lstm_continuous_features": CONTINUOUS_FEATURES,
    "lstm_categorical_features": CATEGORICAL_FEATURES,
    "lgbm_best_params": best_params,
    "lstm_embedding_sizes": {"city": NUM_CITIES, "store": NUM_STORES, "product": NUM_PRODUCTS},
    "lstm_hidden_dim": 64,
    "lstm_epochs_run": epoch,
    "artifacts": {
        "lightgbm_model": "best_lightgbm.pkl",
        "lightgbm_features": "lgbm_features.pkl",
        "lstm_weights": "best_lstm.pth",
        "lstm_scaler": "lstm_scaler.pkl",
        "feature_importance": "lightgbm_feature_importance.csv",
        "final_comparison": "final_model_comparison.csv",
    },
}

with open("run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2)

print("Saved: run_metadata.json")

LightGBM (scored on LSTM's matched test_indices):
MAE: 0.471188  RMSE: 0.727935  R2: 0.713717

FINAL MODEL COMPARISON
                          Model      Split      MAE     RMSE       R2       N
               Baseline - Lag 1       Test 0.599885 0.944151 0.509999 1639185
               Baseline - Lag 7       Test 0.628725 1.008916 0.444282 1612608
       LightGBM (full test set)       Test 0.470513 0.726350 0.709632 1643401
LightGBM (matched to LSTM rows)       Test 0.471188 0.727935 0.713717 1576934
                           LSTM Validation 0.746668 1.036168 0.309024 1215600
                           LSTM       Test 0.856101 1.268904 0.130103 1576934

Saved: final_model_comparison.csv
Saved: run_metadata.json
